In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

In [24]:
df = pd.read_csv("../Dataset/telco_clean.csv")

In [25]:
y = df["Churn"].map({"No": 0, "Yes": 1})
x = df.drop(columns=["customerID", "Churn", "Churn_num"], errors="ignore")

assert "Churn" not in x.columns and "Churn_num" not in x.columns
print(x.shape, y.shape)

(7043, 19) (7043,)


In [26]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]

x["num_services"] = (x[addon_cols] == "Yes").sum(axis=1)

x["tenure_group"] = pd.cut(x["tenure"], bins=[-1, 12, 24, 48, 72],
                           labels=["0-12", "13-24", "25-48", "49-72"],).astype(str)

print(y.groupby(x["num_services"]).mean())
print(y.groupby(x["tenure_group"]).mean())

num_services
0    0.214060
1    0.457557
2    0.358180
3    0.273703
4    0.223005
5    0.124343
6    0.052817
Name: Churn, dtype: float64
tenure_group
0-12     0.474382
13-24    0.287109
25-48    0.203890
49-72    0.095132
Name: Churn, dtype: float64


In [27]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(x_train.shape, x_test.shape, y_train.shape, y_test.shape)
print(y_train.mean(), y_test.mean())

(5634, 21) (1409, 21) (5634,) (1409,)
0.2653532126375577 0.2654364797728886


In [28]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges", "num_services"]

categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod", "tenure_group",
]

# SeniorCitizen is already 0/1, so it will pass through unchanged
print(len(numeric_cols) + len(categorical_cols) +
      1, "should equal", x.shape[1])

21 should equal 21


In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                              sparse_output=False), categorical_cols),
    ],
    remainder="passthrough",
)

X_train_prep = preprocessor.fit_transform(x_train)
X_test_prep = preprocessor.transform(x_test)

print(X_train_prep.shape, X_test_prep.shape)

(5634, 31) (1409, 31)


In [31]:
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names))
print(feature_names)

prep_data = np.asarray(X_train_prep)
prep_df = pd.DataFrame(np.asarray(prep_data), columns=feature_names)
print(prep_df[["num__tenure", "num__MonthlyCharges"]].mean().round(3))
print(prep_df[["num__tenure", "num__MonthlyCharges"]].std().round(3))

31
['num__tenure' 'num__MonthlyCharges' 'num__TotalCharges'
 'num__num_services' 'cat__gender_Male' 'cat__Partner_Yes'
 'cat__Dependents_Yes' 'cat__PhoneService_Yes' 'cat__MultipleLines_Yes'
 'cat__InternetService_DSL' 'cat__InternetService_Fiber optic'
 'cat__InternetService_No' 'cat__OnlineSecurity_Yes'
 'cat__OnlineBackup_Yes' 'cat__DeviceProtection_Yes'
 'cat__TechSupport_Yes' 'cat__StreamingTV_Yes' 'cat__StreamingMovies_Yes'
 'cat__Contract_Month-to-month' 'cat__Contract_One year'
 'cat__Contract_Two year' 'cat__PaperlessBilling_Yes'
 'cat__PaymentMethod_Bank transfer (automatic)'
 'cat__PaymentMethod_Credit card (automatic)'
 'cat__PaymentMethod_Electronic check' 'cat__PaymentMethod_Mailed check'
 'cat__tenure_group_0-12' 'cat__tenure_group_13-24'
 'cat__tenure_group_25-48' 'cat__tenure_group_49-72'
 'remainder__SeniorCitizen']
num__tenure           -0.0
num__MonthlyCharges   -0.0
dtype: float64
num__tenure            1.0
num__MonthlyCharges    1.0
dtype: float64
